# Test for "map of science" for a cherrypicked paper

- parse paper (synthesis, images)
- maybe load the correct image ID
- connect the compound to the synthesis procedure
- some way to plot?


In [ ]:
# Imports
import dotenv
from IPython.display import Markdown, display

from llm_synthesis.transformers.pdf_extraction import MistralPDFExtractor

dotenv.load_dotenv()


with open(
    "/Users/magdalenalederbauer/Code/lematerial-llm-synthesis/results/thermocat-papers/1556-276X-9-254.pdf",
    "rb",
) as f:
    pdf_data = f.read()

pdf_extractor = MistralPDFExtractor(
    structured=False
)  # You can set the api key as an argument or in the environment variable MISTRAL_API_KEY (use a .env file)
mistral_extracted_text = pdf_extractor.forward(pdf_data)

display(Markdown(mistral_extracted_text))

In [ ]:
from llm_synthesis.transformers.synthesis_extraction import (
    DspySynthesisExtractor,
    make_dspy_synthesis_extractor_signature,
)
from llm_synthesis.utils.dspy_utils import get_llm_from_name

# Let's make a signature for the structured data extraction
signature = make_dspy_synthesis_extractor_signature()

lm = get_llm_from_name("gemini-2.0-flash", {"temperature": 0.0})
# Let's make a structured data extractor
structured_data_extractor = DspySynthesisExtractor(signature, lm)

In [ ]:
from llm_synthesis.utils.markdown_utils import remove_figs

publication_text = remove_figs(mistral_extracted_text)

# Let's extract the structured data
structured_data_from_publication_text = structured_data_extractor.forward(
    input=(publication_text, "CeO2 nanofibers")
)

structured_data_from_publication_text

In [ ]:
structured_data_from_publication_text.steps

In [ ]:
from llm_synthesis.transformers.figure_extraction import (
    FigureExtractorMarkdown,
)

figure_extractor = FigureExtractorMarkdown()

figures = figure_extractor.forward(mistral_extracted_text)

print(f"{len(figures)} figures was found in this paper.")

In [ ]:
import base64
import io

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Loop through each figure in the 'figures' list
for index, figure in enumerate(figures):
    print("=" * 80)

    # Determine if the figure is quantitative
    is_quantitative = (
        "Quantitative" if figure.quantitative else "Not Quantitative"
    )

    # Print figure information
    print(
        f"Figure {index + 1}: {is_quantitative} - Figure Class: {figure.figure_class}"
    )

    # Decode Base64 image data and open it using PIL
    image_data = base64.b64decode(figure.base64_data)
    image_stream = io.BytesIO(image_data)
    image = Image.open(image_stream)

    # Convert image to NumPy array for visualization
    image_array = np.array(image)

    # Plot the image using Matplotlib
    plt.imshow(image_array)
    plt.axis("off")  # Hide axes for better visual appearance
    plt.title(f"{is_quantitative}: {figure.figure_class}")
    plt.show()

In [ ]:
# Let's print the first figure

import base64

from IPython.display import Image

# Convert base64 string to image and display
Image(base64.b64decode(figures[0].base64_data))

In [ ]:
from llm_synthesis.models.figure import FigureInfoWithPaper
from llm_synthesis.transformers.figure_description import (
    DspyFigureDescriptionExtractor,
    make_dspy_figure_description_extractor_signature,
)
from llm_synthesis.utils.dspy_utils import get_llm_from_name
from llm_synthesis.utils.markdown_utils import remove_figs

# Let's make a signature for the figure description extraction
signature = make_dspy_figure_description_extractor_signature(
    signature_name="DspyFigureDescriptionExtractorSignature",
    instructions="Extract the figure description from the figure.",
    publication_text_description="The publication text to extract the figure description from.",
    si_text_description="The supporting information text to extract the figure description from.",
    figure_base64_description="The base64 encoded image of the figure to extract the description from.",
    caption_context_description="The text context surrounding the figure position including the figure caption and nearby paragraphs that reference this figure.",
    figure_position_info_description="The information about the figure's position in the document (e.g., 'Figure 2', 'Fig. 3a', 'Scheme 1') to help with contextual understanding.",
    figure_description_description="The extracted figure description.",
)

lm = get_llm_from_name("gpt-4o-mini", {"temperature": 0.0})
paper_name = ""
# with open("../data/txt_papers/docling/test_" + paper_name + ".md", errors="replace") as f:
with open(
    "../data/txt_papers/mistral/test_" + paper_name + ".md", errors="replace"
) as f:
    publication_text = f.read()

publication_text = remove_figs(publication_text)

figure_info_with_paper = FigureInfoWithPaper(
    **figures[2].__dict__,
    paper_text=publication_text,
    si_text="",
)

figure_description_extractor = DspyFigureDescriptionExtractor(signature, lm)

figure_description = figure_description_extractor.forward(
    figure_info_with_paper
)

figure_description